# Datenlabor Tag 8 · Kranz Werkzeughandel
## Teil 1 · Finden: Probleme in `bestellungen_A.csv` suchen und zählen

**Ziel dieses Teils:** nur **zählen und anschauen**, noch **nichts reparieren**. Am Ende steht eine Tabelle *Problemart, Anzahl, Beispielzeilen*.

**Arbeitsweise:** Ein kleines Skript durchsucht alle 200 Zeilen mit **Mustern** (z. B. "Datum sieht aus wie `99.99.9999`"). So sind die Zahlen **exakt und wiederholbar**. Wo ein Muster nicht reicht, schaue ich mir die betroffenen Zeilen selbst an. Das ist im Text jeweils als *Meine Lesart* gekennzeichnet.

Die Datei hat 200 Zeilen, davon ist eine komplett leer (Abschnitt 1). Alle Zellen laufen von oben nach unten (`Shift+Enter`). Die Daten sind fiktiv.

## 0 · Laden

In [ ]:
import re                      # reguläre Ausdrücke: Muster in Texten suchen
import unicodedata             # Umlaute und Akzente vergleichbar machen
import pandas as pd            # Tabellen
from pathlib import Path

pd.set_option("display.max_rows", 200)       # lange Tabellen nicht abschneiden
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

ORDNER = Path.cwd()            # das Notebook liegt im selben Ordner wie die CSV-Dateien

**Wichtig beim Laden:** Alle Spalten werden als **Text** gelesen (`dtype=str`), leere Zellen bleiben `""` (`keep_default_na=False`). Sonst würde pandas z. B. aus der PLZ `04109` die Zahl `4109` machen und den Fehler **selbst erzeugen**, den wir suchen.

**Zum Encoding:** Beide Dateien beginnen mit den Bytes `EF BB BF`. Das ist ein **BOM** (Byte Order Mark, ein unsichtbares Zeichen am Dateianfang, das Excel beim Speichern als UTF-8 setzt). Wir prüfen gleich, was passiert, wenn man es nicht beachtet.

In [ ]:
# Die ersten drei Bytes der Datei anschauen
rohbytes = (ORDNER / "bestellungen_A.csv").read_bytes()[:3]
print("Erste Bytes:", rohbytes, "-> BOM" if rohbytes == b"\xef\xbb\xbf" else "-> kein BOM")

# Variante 1: so wie die Vorlage regeln.py liest (encoding="utf-8")
vorlage = pd.read_csv(ORDNER / "bestellungen_A.csv", sep=";", dtype=str,
                      keep_default_na=False, encoding="utf-8")
print("Spaltennamen mit utf-8      :", list(vorlage.columns)[:3], "...")

# Variante 2: BOM wird entfernt (encoding="utf-8-sig")
b = pd.read_csv(ORDNER / "bestellungen_A.csv", sep=";", dtype=str,
                keep_default_na=False, encoding="utf-8-sig")
stamm = pd.read_csv(ORDNER / "kundenstamm.csv", sep=";", dtype=str,
                    keep_default_na=False, encoding="utf-8-sig")
print("Spaltennamen mit utf-8-sig  :", list(b.columns)[:3], "...")
print(f"Zeilen in der Datei: {b.shape[0]}, Kundenstamm: {stamm.shape[0]} Kunden")

**Fund 0:** Kein Problem. Die pandas-Version hier entfernt das BOM auch bei `encoding="utf-8"` selbst, die Spalte heißt in beiden Varianten `zeile`. Die Vorlage `regeln.py` kann also so lesen, wie sie es tut. (Mit dem `csv`-Modul würde das BOM dagegen im ersten Spaltennamen stehen bleiben: `'\ufeffzeile'`. Deshalb prüfe ich es und gehe nicht davon aus.)

## 1 · Überblick: Leere Zeilen und leere Zellen
Leer heißt: nach Entfernen von Leerzeichen bleibt nichts übrig. Eine Zelle nur mit Leerzeichen zählt als leer.
Zuerst die Frage: Gibt es **komplett leere Zeilen** (alle Zellen leer)?

In [ ]:
ist_leer = b.apply(lambda s: s.str.strip() == "")       # True/False je Zelle
komplett_leer = ist_leer.all(axis=1)                    # Zeile, in der ALLE Zellen leer sind
n_leerzeilen = int(komplett_leer.sum())
print("Komplett leere Zeilen:", n_leerzeilen, "-> Index in der Tabelle:", list(b.index[komplett_leer]))
print("Letzte Zeile der Datei, roh:", b.iloc[-1].tolist())

# Ab hier arbeiten wir mit den Zeilen, die überhaupt etwas enthalten.
# Die Leerzeile ist ein Format-Fehler (Trennzeile am Dateiende), kein Datenfehler.
b = b[~komplett_leer].copy()
ist_leer = ist_leer[~komplett_leer]
print(f"\nWeiter mit {len(b)} Bestellungen.\n")

print("Leere Zellen pro Spalte (ohne die Leerzeile):")
print(ist_leer.sum().to_string())
print()
print("Zeilen mit mindestens einer leeren Zelle:", int(ist_leer.any(axis=1).sum()))
b[ist_leer.any(axis=1)]

## 2 · Das Muster-Werkzeug
Um Formate zu zählen, ersetzen wir jede **Ziffer durch `9`** und jeden **Buchstaben durch `a`**. So wird `02.01.2025` zu `99.99.9999` und `192,17 €` zu `999,99 €`. Gleiche Muster bedeuten gleiches Format, egal welche Zahl drinsteht.

In [ ]:
def muster(text):
    """Wandelt Ziffern in 9 und Buchstaben in a um: nur die Form bleibt."""
    text = re.sub(r"\d", "9", text)
    text = re.sub(r"[A-Za-zÄÖÜäöüß]", "a", text)
    return text

def zaehle_muster(spalte, tabelle=b):
    """Zählt, wie oft jedes Muster in einer Spalte vorkommt, mit Beispiel und Zeilen-IDs."""
    t = tabelle.assign(muster=tabelle[spalte].map(muster))
    gruppen = t.groupby("muster").agg(
        anzahl=("zeile", "size"),
        beispiel=(spalte, "first"),
        zeilen=("zeile", lambda z: ", ".join(z.head(4)) + (" ..." if len(z) > 4 else "")),
    )
    return gruppen.sort_values("anzahl", ascending=False)

print(muster("02.01.2025"), muster("192,17 €"), muster("1.250,80"))

## 3 · `bestelldatum`
Erwartet: `JJJJ-MM-TT`, z. B. `2025-03-14`.

In [ ]:
zaehle_muster("bestelldatum")

**Was die Muster zeigen** (zuordnen, nicht raten):
- `9999-99-99` ist das **Zielformat** (ISO).
- `99.99.9999` ist **deutsch** (Tag.Monat.Jahr).
- `99/99/9999` ist **mehrdeutig**: Tag/Monat oder Monat/Tag? Siehe unten.
- `99.99.99` hat nur **zweistelliges Jahr**.
- `aaaa 9999`, `aaaaa 9999`, `aaaaaaa 9999` sind **Monatsname plus Jahr**: Der **Tag fehlt**.
- Leere Zelle.

### 3a · Ist `99/99/9999` Tag/Monat oder Monat/Tag?
Wir suchen Zeilen, bei denen **eine der beiden Zahlen größer als 12** ist. Diese Zahl kann kein Monat sein und verrät die Reihenfolge.

In [ ]:
slash = b[b.bestelldatum.str.match(r"^\d\d/\d\d/\d{4}$")].copy()
teile = slash.bestelldatum.str.split("/", expand=True).astype(int)
slash["erste"], slash["zweite"] = teile[0], teile[1]
slash["erste_ueber_12"] = slash.erste > 12     # erste Zahl kann kein Monat sein -> Tag/Monat
slash["zweite_ueber_12"] = slash.zweite > 12   # zweite Zahl kann kein Monat sein -> Monat/Tag
print("Erste Zahl > 12  (also Tag/Monat):", slash.erste_ueber_12.sum())
print("Zweite Zahl > 12 (also Monat/Tag):", slash.zweite_ueber_12.sum())
print("Weder noch (nicht entscheidbar)  :", (~slash.erste_ueber_12 & ~slash.zweite_ueber_12).sum())
slash[["zeile", "quelle", "bestelldatum", "erste", "zweite"]]

**Meine Lesart:** Wenn **nur eine** der beiden Seiten größer als 12 vorkommt, ist für *diese Quelle* die Reihenfolge belegt. Dazu passt der Hinweis aus dem Aufgabenblatt: Herr Hoffmann kam von einem US-Hersteller. Ob **alle** `99/99/9999` aus derselben Quelle stammen, prüfen wir jetzt.

In [ ]:
slash.groupby("quelle").agg(anzahl=("zeile", "size"),
                            erste_ueber_12=("erste_ueber_12", "sum"),
                            zweite_ueber_12=("zweite_ueber_12", "sum"))

### 3b · Zweistelliges Jahr (`99.99.99`) und Monatsnamen

In [ ]:
print("Zweistelliges Jahr:")
display(b[b.bestelldatum.str.match(r"^\d\d\.\d\d\.\d\d$")][["zeile", "quelle", "bestelldatum"]])
print("Monatsname statt Tag:")
display(b[b.bestelldatum.str.match(r"^[A-Za-zÄÖÜäöüß]+ \d{4}$")][["zeile", "quelle", "bestelldatum", "kunde", "betrag"]])

**Meine Lesart:** Bei `99.99.99` ist die Reihenfolge Tag.Monat.Jahr (Punkt = deutsch), das Jahr muss auf `20xx` ergänzt werden. Bei **Monat + Jahr** fehlt der **Tag**: Er lässt sich **nicht sicher** ergänzen, die Zeile sollte einen **Hinweis** bekommen (der Monat reicht für eine Monatsauswertung, aber nicht für das Format `JJJJ-MM-TT`).

## 4 · `betrag`
Erwartet: Euro als Zahl mit **Dezimalpunkt**, ohne Währungszeichen, z. B. `1250.80`.

In [ ]:
zaehle_muster("betrag")

**Was die Muster zeigen:**
- `999.99`: Zielformat (Punkt als Dezimalzeichen).
- `999,99 €`, `999,99`, `99,99 €`: **Komma** als Dezimalzeichen, teils mit **€**.
- `999,99 aaa`, `999 aaa`: Währung als **Text** (z. B. `EUR`) angehängt.
- `9,999.99`, `99,999.99`: **Tausenderkomma** plus Dezimalpunkt (US-Schreibweise).
- `99.999,99 €`: **Tausenderpunkt** plus Dezimalkomma (deutsche Schreibweise).
- `-999.99`: **negativ**: Gutschrift?
- Leere Zelle.

**Falle:** Ein einzelnes Trennzeichen ist mehrdeutig. `1.250` ist im Deutschen 1250, im Englischen 1,25. Bei `9,999.99` oder `99.999,99` sind **beide** Zeichen da, und das **letzte** ist das Dezimalzeichen.

In [ ]:
print("Betrag nicht im Zielformat (Ziffern, Punkt, höchstens zwei Dezimalstellen):")
ziel = r"^-?\d+\.\d{2}$"
nicht_ziel = b[(b.betrag.str.strip() != "") & ~b.betrag.str.match(ziel)]
print(len(nicht_ziel), "von", len(b))
print()
print("Text-Währungen:", sorted(set(re.findall(r"[A-Za-z]+", " ".join(b.betrag)))))
print()
print("Negative Beträge:")
b[b.betrag.str.startswith("-")][["zeile", "quelle", "kunde", "bestelldatum", "betrag"]]

**Meine Lesart:** Ein Webshop erfasst Gutschriften als Bestellung mit negativem Betrag (Aufgabenblatt). Ein negativer Betrag ist also **kein Fehler**. Die Regel darf ihn **nicht verändern** (auch nicht den Betrag in den Absolutwert drehen). Wichtig ist, dass spätere Umsatzrechnungen Gutschriften **abziehen**. Ob dieser Betrag aus dem Webshop kommt, sieht man in der Spalte `quelle`.

## 5 · `plz` und `ort`
Erwartet: PLZ **fünfstellig**, Ort **genau so wie im Kundenstamm**.

In [ ]:
zaehle_muster("plz")

In [ ]:
# Vier-stellige PLZ: Kommen sie aus Sachsen/Thüringen (führende 0)?
kurz = b[b.plz.str.match(r"^\d{4}$")]
print(len(kurz), "Zeilen mit vierstelliger PLZ")
print(kurz.groupby("ort").size().sort_values(ascending=False).head(15).to_string())

**Meine Lesart:** Vierstellige PLZ sind **PLZ mit abgeschnittener führender Null**: `4109` ist `04109` (Leipzig), `1067` ist `01067` (Dresden). Das passiert, wenn Excel die PLZ als Zahl speichert. Man erkennt die Fälle an den Orten (alle aus den neuen Bundesländern). Die Regel "führende `0` ergänzen" ist nur **sicher**, wenn die PLZ 4 Ziffern hat.

### 5a · `ort`: Auffälligkeiten

In [ ]:
ort_muster = zaehle_muster("ort")
ort_muster

In [ ]:
# Ort mit PLZ davor ("36381 Schlüchtern")
plz_im_ort = b[b.ort.str.match(r"^\d{5} ")]
print("PLZ im Ortsfeld:", len(plz_im_ort))
display(plz_im_ort[["zeile", "plz", "ort"]].head(12))

# Kaputte Umlaute (Mojibake): UTF-8-Text, der als Latin-1 gelesen wurde -> "Ã¶" statt "ö"
# Wir prüfen ALLE Textspalten, nicht nur den Ort.
kaputt = b.apply(lambda s: s.str.contains("Ã|â€"))
print("Zellen mit kaputten Umlauten je Spalte:")
print(kaputt.sum()[kaputt.sum() > 0].to_string())
display(b[kaputt.any(axis=1)][["zeile", "quelle", "kunde", "plz", "ort"]])

## 6 · `kunde`: Abgleich mit dem Kundenstamm
Erwartet: Name **genau so wie im Kundenstamm**. Wir prüfen in drei Stufen: (1) genau gleich, (2) gleich nach Normalisieren, (3) gar nicht auffindbar.

In [ ]:
namen_stamm = set(stamm.kunde)
exakt = b.kunde.isin(namen_stamm)
print(f"(1) Name steht exakt im Kundenstamm: {exakt.sum()} von {len(b)}")
print(f"    Name weicht ab:                 {(~exakt).sum()}")

**Normalisieren** heißt: Schreibweisen vergleichbar machen, ohne den Namen inhaltlich zu verändern: Groß/Klein ignorieren, Leerzeichen und Sonderzeichen vereinheitlichen. Das **Ergebnis ist nur ein Vergleichsschlüssel**, der Name in der Tabelle wird später durch den **Namen aus dem Kundenstamm** ersetzt.

In [ ]:
def schluessel(name):
    """Vergleichsschlüssel: Kleinschreibung, nur Buchstaben und Ziffern, ein Leerzeichen dazwischen."""
    n = unicodedata.normalize("NFKC", name).casefold()
    n = n.replace("&", " und ")
    n = re.sub(r"[^0-9a-zäöüß]+", " ", n)      # Bindestriche, Punkte, Kommas -> Leerzeichen
    return re.sub(r"\s+", " ", n).strip()

stamm["schluessel"] = stamm.kunde.map(schluessel)
b["schluessel"] = b.kunde.map(schluessel)

# Wie viele Namen im Stamm sind nach Normalisieren nicht mehr eindeutig?
doppelt = stamm[stamm.schluessel.duplicated(keep=False)].sort_values("schluessel")
print("Stamm: gleicher Schlüssel bei verschiedenen Kundennummern:", len(doppelt))
display(doppelt)

**Fund:** Im Kundenstamm gibt es **denselben Namen mit zwei Kundennummern**. Das sind die zwei **Betriebe** eines Kunden (Aufgabenblatt: *Elektro Weber hat zwei Betriebe: Kassel und Baunatal*). Der Name allein reicht dann nicht, die **PLZ/Ort** muss mitentscheiden.

In [ ]:
stufe2 = b.schluessel.isin(set(stamm.schluessel))
print(f"(1) exakt gleich                         : {exakt.sum()}")
print(f"(2) gleich nach Normalisieren (neu dazu) : {(stufe2 & ~exakt).sum()}")
print(f"(3) auch dann nicht gefunden             : {(~stufe2).sum()}")
print()
print("Schreibweisen, die nur durch Normalisieren passen (je Beispiel):")
x = b[stufe2 & ~exakt][["zeile", "kunde", "schluessel"]]
x = x.merge(stamm[["schluessel", "kunde"]].drop_duplicates("schluessel"), on="schluessel", suffixes=(" (Bestellung)", " (Stamm)"))
x[["zeile", "kunde (Bestellung)", "kunde (Stamm)"]].head(20)

In [ ]:
print("Nicht gefunden, auch nach Normalisieren:")
rest = b[~stufe2][["zeile", "quelle", "kunde", "plz", "ort"]]
print(len(rest), "Zeilen")
rest

**Meine Lesart:** Hier trennt sich "sicher" von "unsicher". Beispiele, die ich mir anschaue: fehlt nur die **Rechtsform** (`GmbH`, `e.K.`)? Dann gehört die Zeile mit hoher Wahrscheinlichkeit zu einem Kunden, ist aber nur **sicher**, wenn es **genau einen** Stamm-Kunden gibt, der so beginnt. Gibt es mehrere, bekommt die Zeile einen **Hinweis** statt einer Korrektur.

In [ ]:
# Rechtsformen aus dem Schlüssel entfernen, um "ohne Rechtsform" zu testen
RECHTSFORMEN = r"\b(gmbh und co kg|gmbh|e k|ek|kg|ohg|ag|ug)\b"
def kern(s):
    return re.sub(RECHTSFORMEN, "", s).strip()

stamm["kern"] = stamm.schluessel.map(kern)
b["kern"] = b.schluessel.map(kern)

# Pro Kern: wie viele verschiedene Stamm-Kunden (Kundennummern)?
kern_zu_nr = stamm.groupby("kern").kundennr.nunique()

rest = b[~stufe2].copy()
rest["treffer_im_stamm"] = rest.kern.map(kern_zu_nr).fillna(0).astype(int)
print("Von den nicht gefundenen Zeilen:")
print("  Kern passt zu genau 1 Kundennummer :", (rest.treffer_im_stamm == 1).sum())
print("  Kern passt zu 2+ Kundennummern     :", (rest.treffer_im_stamm >= 2).sum())
print("  Kern passt zu keinem Kunden        :", (rest.treffer_im_stamm == 0).sum())
rest[["zeile", "kunde", "plz", "ort", "treffer_im_stamm"]]

## 7 · `quelle` und `artikelgruppe`
Beide sollen **unverändert** übernommen werden. Wir suchen nur, ob sie sauber sind.

In [ ]:
for spalte in ["quelle", "artikelgruppe"]:
    print(f"--- {spalte}")
    t = b[spalte].value_counts(dropna=False)
    print(t.to_string())
    print("Werte mit Leerzeichen am Rand:", (b[spalte] != b[spalte].str.strip()).sum())
    print()

## 8 · Leerzeichen am Rand (alle Spalten)
Leerzeichen vorn oder hinten sind **unsichtbar**, machen aber `Elektro Weber GmbH` und ` Elektro Weber GmbH` zu verschiedenen Texten.

In [ ]:
rand = pd.DataFrame({sp: (b[sp] != b[sp].str.strip()) for sp in b.columns if sp != "schluessel"})
rand_summe = rand.sum()
print(rand_summe[rand_summe > 0].to_string())
b[rand.any(axis=1)][["zeile", "kunde", "ort"]].head(10)

## 9 · Dubletten: Dieselbe Bestellung zweimal?
Dubletten sind **leise Fehler**: Kein Wert ist kaputt, aber der Umsatz wird doppelt gezählt. Wir brauchen **normalisierte** Werte, sonst finden wir sie nicht (`04.01.2025` und `2025-01-04` sind dasselbe Datum).

**Achtung, Falle aus dem Aufgabenblatt:** *Die Schreinerei Ott bestellt jeden Monat dieselben Schleifmittel.* Gleicher Kunde und gleicher Betrag **in verschiedenen Monaten** ist also **keine** Dublette. Wir vergleichen deshalb **Kunde + Datum + Betrag**.

In [ ]:
# Hilfsfunktionen, die nur zum Vergleichen dienen (die Tabelle b selbst bleibt unverändert)
def datum_iso(text, slash_reihenfolge=None):
    """Gibt JJJJ-MM-TT zurück oder None, wenn das Datum nicht sicher bestimmbar ist."""
    t = text.strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", t):
        return t
    m = re.fullmatch(r"(\d{2})\.(\d{2})\.(\d{4})", t)
    if m:
        return f"{m[3]}-{m[2]}-{m[1]}"
    m = re.fullmatch(r"(\d{2})\.(\d{2})\.(\d{2})", t)
    if m:
        return f"20{m[3]}-{m[2]}-{m[1]}"
    m = re.fullmatch(r"(\d{2})/(\d{2})/(\d{4})", t)
    if m and slash_reihenfolge == "TM":
        return f"{m[3]}-{m[2]}-{m[1]}"
    if m and slash_reihenfolge == "MT":
        return f"{m[3]}-{m[1]}-{m[2]}"
    return None            # Monatsname, leer, oder Slash ohne bekannte Reihenfolge

def betrag_zahl(text):
    """Wandelt einen Betrag in eine Zahl um: das LETZTE Trennzeichen ist das Dezimalzeichen."""
    t = re.sub(r"[^\d,.\-]", "", text)         # € und Text-Währung entfernen
    if t == "":
        return None
    letztes = max(t.rfind(","), t.rfind("."))
    if letztes == -1:
        return float(t)
    ganz = re.sub(r"[,.]", "", t[:letztes])
    return float(f"{ganz}.{t[letztes + 1:]}")

b["datum_i"] = b.bestelldatum.map(lambda t: datum_iso(t, slash_reihenfolge="MT"))   # MT = Monat/Tag (Abschnitt 3a)
b["betrag_z"] = b.betrag.map(betrag_zahl)
print("Datum lesbar:", b.datum_i.notna().sum(), "von", len(b))
print("Betrag lesbar         :", b.betrag_z.notna().sum(), "von", len(b))

**Zur Reihenfolge bei `99/99/9999`:** Abschnitt 3a belegt **Monat/Tag/Jahr** (`slash_reihenfolge="MT"`), und zwar nur bei `AD Hoffmann`. Das ist *meine Lesart aus den Daten*: 11 von 15 Werten sind eindeutig (die zweite Zahl ist größer als 12), 4 wären für sich nicht entscheidbar, kommen aber aus derselben Quelle. Datumswerte, die auch so nicht lesbar sind (Monatsname, leer), fehlen in der Dublettensuche.

In [ ]:
# Dublettenschlüssel: gleicher Kunde (Schlüssel), gleiches Datum, gleicher Betrag
mit_schluessel = b.dropna(subset=["datum_i", "betrag_z"])
dub = mit_schluessel[mit_schluessel.duplicated(["schluessel", "datum_i", "betrag_z"], keep=False)] \
        .sort_values(["schluessel", "datum_i", "betrag_z"])
print(len(dub), "Zeilen gehören zu", dub.groupby(["schluessel", "datum_i", "betrag_z"]).ngroups, "Dubletten-Gruppen")
dub[["zeile", "quelle", "kunde", "bestelldatum", "betrag", "plz", "ort"]]

In [ ]:
# Wie sind die Gruppen aufgebaut: gleiche Quelle oder verschiedene?
g = dub.groupby(["schluessel", "datum_i", "betrag_z"]).agg(
    zeilen=("zeile", lambda z: ", ".join(z)),
    quellen=("quelle", lambda q: " + ".join(q)),
)
g

**Meine Lesart:** Eine Gruppe aus `Webshop` + `AD …` ist das Muster **"Außendienst trägt eine Webshop-Bestellung zusätzlich ein"**. Das Aufgabenblatt sagt: nach Datum und Betrag gleiche Bestellung = dieselbe. Ob eine Gruppe aus **zwei Außendienstlern** oder **zweimal Webshop** (Doppelklick) eine Dublette ist oder eine echte Wiederholung, entscheidet die Fachlichkeit. Die Regel sollte nur **sichere** Fälle löschen und den Rest **markieren**.

In [ ]:
# Gegenprobe: Wiederholungen derselben Beträge bei gleichem Kunden in VERSCHIEDENEN Monaten (die "Ott-Falle")
b["monat"] = b.datum_i.str[:7]
wdh = b.dropna(subset=["monat", "betrag_z"])
wdh = wdh[wdh.duplicated(["schluessel", "betrag_z"], keep=False)]
wdh = wdh[~wdh.zeile.isin(dub.zeile)]
print("Gleicher Kunde + gleicher Betrag, aber verschiedenes Datum:", len(wdh), "Zeilen")
wdh.sort_values(["schluessel", "datum_i"])[["zeile", "quelle", "kunde", "bestelldatum", "betrag", "artikelgruppe"]]

## 10 · Zusammenfassung Finden
Die Zahlen unten werden **aus den Daten berechnet** (nicht abgetippt), damit sie bei jedem Lauf stimmen.

In [ ]:
datum_muster = b.bestelldatum.map(muster)
betrag_m = b.betrag.map(muster)
ort_dazu = b.ort.str.match(r"^\d{5} ")

befunde = [
    ("Zeile komplett leer (nur Trennzeichen)", "alle", n_leerzeilen),
    ("Datum: Punkt-Format (TT.MM.JJJJ)", "bestelldatum", int((datum_muster == "99.99.9999").sum())),
    ("Datum: Schrägstrich (TT/MM oder MM/TT)", "bestelldatum", int((datum_muster == "99/99/9999").sum())),
    ("Datum: zweistelliges Jahr", "bestelldatum", int((datum_muster == "99.99.99").sum())),
    ("Datum: nur Monatsname + Jahr", "bestelldatum", int(datum_muster.str.match(r"^a+ 9999$").sum())),
    ("Datum: leer", "bestelldatum", int((b.bestelldatum.str.strip() == "").sum())),
    ("Betrag: nicht im Zielformat (ohne leere)", "betrag", int(((b.betrag.str.strip() != "") & ~b.betrag.str.match(r"^-?\d+\.\d{2}$")).sum())),
    ("Betrag: leer", "betrag", int((b.betrag.str.strip() == "").sum())),
    ("Betrag: Währung (€ oder Text)", "betrag", int(b.betrag.str.contains(r"[€A-Za-z]").sum())),
    ("Betrag: negativ (Gutschrift?)", "betrag", int(b.betrag.str.startswith("-").sum())),
    ("PLZ: vierstellig", "plz", int(b.plz.str.match(r"^\d{4}$").sum())),
    ("PLZ: leer", "plz", int((b.plz.str.strip() == "").sum())),
    ("Ort: PLZ im Ortsfeld", "ort", int(ort_dazu.sum())),
    ("Ort: kaputte Umlaute", "ort", int(b.ort.str.contains("Ã|â€").sum())),
    ("Kunde: kaputte Umlaute", "kunde", int(b.kunde.str.contains("Ã|â€").sum())),
    ("Ort: leer", "ort", int((b.ort.str.strip() == "").sum())),
    ("Kunde: Name weicht vom Stamm ab", "kunde", int((~exakt).sum())),
    ("Kunde: auch normalisiert nicht gefunden", "kunde", int((~stufe2).sum())),
    ("Dubletten (Kunde + Datum + Betrag)", "mehrere", int(len(dub))),
]
tabelle = pd.DataFrame(befunde, columns=["Problem", "Spalte", "Anzahl"])
tabelle[f"Anteil von {len(b)}"] = (tabelle.Anzahl / len(b)).map("{:.1%}".format)
tabelle

### Teil 1 abgeschlossen
Das Bild: Fast jede Spalte hat **mehrere** Schreibweisen für dasselbe. Die meisten Probleme sind **laut** (man sieht sie), die Dubletten sind **leise**. Weiter mit Teil 2: **Regeln**.

*Hinweis zu den Dubletten:* Hier haben wir nur **4 Paare** gefunden, weil die Kundennamen noch nicht angeglichen sind. Nach Regel 7 (Kundenname) sind es **6 Paare**. Zwei Außendienst-Zeilen hatten den Namen ohne `GmbH` / `e.K.` geschrieben und fielen vorher nicht auf.

---
# Teil 2 · Regeln: Aus den Befunden werden Bauanleitungen

## Wie Regeln gebaut sind
Eine **Regelkarte** hat drei Teile. Sie lassen sich als Satz lesen:

> **Wenn** *(woran man es erkennt)*, **dann** *(was das Skript tut)*, **außer / sonst markieren** *(wann es die Finger davon lässt)*.

Vier Spielregeln, nach denen ich die Regeln geschrieben habe:

1. **Nur korrigieren, wenn es sicher ist.** Bei Zweifel bleibt der Wert **unverändert** und die Spalte `hinweis` nennt den Grund. Ein falsch "korrigierter" Wert ist schlimmer als ein markierter.
2. **Allgemein formulieren, nicht für einzelne Zeilen.** Die Regel sagt nicht "Zeile A-0014 ist ein Datum von Hoffmann", sondern "wenn eine Zahl über 12 vorkommt, ist es der Tag". Nur so funktioniert sie auch auf den Daten aus dem zweiten Halbjahr, die wir nicht kennen (neue Außendienstler, neue Schreibweisen).
3. **Die Reihenfolge zählt.** Dubletten erkennt man erst, wenn Datum, Betrag und Kunde vereinheitlicht sind. Deshalb steht diese Regel **ganz hinten**.
4. **Alles ist nachvollziehbar.** Das Skript ändert nie die Spalte `zeile`, und der Änderungsbericht zeigt jede Änderung.

## Die zehn Regelkarten

| Nr. | Wenn … (woran erkennbar) | Dann … | Außer / sonst markieren |
|:--:|---|---|---|
| **1** | Eine Zeile ist in **allen** Spalten leer (oder nur Export-Müll wie `,,,,`) | Zeile **löschen** | Hat die Zeile eine echte Nummer wie `B-0123`: bleibt |
| **2** | Ein Text beginnt/endet mit **Leerzeichen**, hat doppelte oder endet auf **`,,,`** | Leerzeichen und Kommas am Ende entfernen | Spalte `zeile` wird nie angefasst |
| **3** | Ein Text enthält **`Ã`** (z. B. `GÃ¶ttingen`) | zurückübersetzen: `Göttingen` | geht es nicht: **markieren** |
| **4** | `bestelldatum` ist **nicht** `JJJJ-MM-TT` | in `JJJJ-MM-TT` umschreiben. Punkt = Tag.Monat.Jahr. `25` als Jahr = `2025`. Schrägstrich: eine **Zahl über 12** verrät den Tag, sonst gilt die Reihenfolge, die die **Quelle** sonst benutzt | Reihenfolge **unklar**, **nur Monat** ohne Tag, nicht lesbar, Datum gibt es nicht: **markieren** |
| **5** | `betrag` ist **nicht** "Zahl mit Punkt und 2 Stellen" | `€` und `EUR` weg, Komma zu Punkt, Tausenderzeichen weg. **Minus bleibt** (Gutschrift) | Betrag **leer** oder Trennzeichen **nicht eindeutig**: **markieren** |
| **6** | Der **Ort** beginnt mit einer PLZ (`36043 Fulda`), oder die **PLZ hat 4 Ziffern** | PLZ in die PLZ-Spalte, Ort ohne PLZ. Bei 4 Ziffern eine **`0` davor** (`4109` → `04109`) | PLZ im Ort **widerspricht** der PLZ-Spalte, oder PLZ hat danach nicht 5 Ziffern: **markieren** |
| **7** | Der **Kundenname** steht nicht genau so im Kundenstamm | Namen des Kundenstamms übernehmen, **wenn genau ein Kunde** passt (gleich nach Groß/Klein und Satzzeichen, oder gleich ohne `GmbH` / `e.K.`). Bei zwei Betrieben mit gleichem Namen entscheiden PLZ und Ort | **kein** oder **mehrere** Treffer: **markieren**. Ähnlich klingende Namen (Ott / Otto) **nie** automatisch zusammenlegen, nur als Vorschlag in den Hinweis |
| **8** | Kunde ist eindeutig, aber **PLZ oder Ort sind leer** | aus dem Kundenstamm einsetzen. Ort nur anders geschrieben (`GÖTTINGEN`): Schreibweise des Stamms | steht ein **anderer** Wert drin, oder der Stamm ist selbst leer, oder der Kunde ist unbekannt: **markieren** |
| **9** | **Zwei Zeilen** haben denselben Kunden, dasselbe Datum, denselben Betrag | ist **genau eine** aus dem **Webshop**: die Außendienst-Zeile **löschen** | sonst **nichts löschen**, die spätere Zeile **markieren**. Anderes Datum, gleicher Betrag (Schreinerei Ott, jeden Monat) ist **keine** Dublette |
| **10** | Das Skript hat Hilfsspalten angelegt | Hilfsspalte **entfernen** | – |

### Warum diese Reihenfolge?
**1 → 2 → 3** machen die Texte sauber (leere Zeilen weg, Leerzeichen weg, Umlaute repariert). **4 → 5 → 6** bringen Datum, Betrag und PLZ ins Zielformat. **7 → 8** gleichen mit dem Kundenstamm ab (braucht saubere PLZ für die zwei Weber-Betriebe). **9** sucht Dubletten, erst jetzt sind sie als gleich erkennbar. **10** räumt auf.

## Vorbereitung: Daten laden und Werkzeuge

In [ ]:
# Die Rohdaten frisch laden: genau so, wie es die Vorlage regeln.py auch tut.
def lade(datei):
    return pd.read_csv(datei, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")

def markiere(df, maske, grund):
    """Schreibt grund in die Spalte hinweis aller Zeilen, für die maske True ist (aus der Vorlage)."""
    alt = df.loc[maske, "hinweis"]
    df.loc[maske, "hinweis"] = alt.where(alt == "", alt + " | ") + grund
    return df

roh = lade(ORDNER / "bestellungen_A.csv")      # alle 200 Zeilen, unverändert
stamm = lade(ORDNER / "kundenstamm.csv")
df = roh.copy()                                 # an df arbeiten wir, roh bleibt unberührt
df["hinweis"] = ""                              # neue Spalte: Grund, wenn etwas nicht sicher ist
print(len(df), "Zeilen, Spalten:", list(df.columns))

In [ ]:
def aenderungen_zwischen(vorher, nachher):
    """Vergleicht zwei Tabellen zeilenweise über die Spalte zeile und liefert jede geänderte Zelle."""
    v, n = vorher.set_index("zeile"), nachher.set_index("zeile")
    geloescht = [z for z in v.index if z not in n.index]
    liste = []
    for spalte in [c for c in n.columns if c in v.columns]:
        alt = v.loc[n.index, spalte].astype(str)
        neu = n[spalte].astype(str)
        for z in alt.index[alt != neu]:
            liste.append({"zeile": z, "spalte": spalte, "vorher": alt[z], "nachher": neu[z]})
    return pd.DataFrame(liste, columns=["zeile", "spalte", "vorher", "nachher"]), geloescht

def anwenden(regel, beispiele=6):
    """Wendet eine Regel auf df an und zeigt, was sie getan hat."""
    global df
    vorher = df.copy()
    df = regel(df, stamm)
    diff, geloescht = aenderungen_zwischen(vorher, df)
    print(f"{regel.__name__}: {len(geloescht)} Zeile(n) gelöscht, {len(diff)} Zelle(n) geändert")
    if len(diff):
        print(diff.groupby("spalte").size().to_string())
        # repr() zeigt auch unsichtbare Leerzeichen am Rand (' text ' statt text)
        zeig = diff.groupby("spalte").head(beispiele).reset_index(drop=True)
        display(zeig.assign(vorher=zeig.vorher.map(repr), nachher=zeig.nachher.map(repr)))
    if geloescht:
        print("Gelöscht:", geloescht)

Die Hilfsfunktionen, die mehrere Regeln gemeinsam benutzen. Jede hat eine kurze Beschreibung. Wer es nur grob verstehen will, kann sie überspringen und bei den Regeln weiterlesen.

In [ ]:
import datetime
import difflib
import re
import unicodedata

RECHTSFORMEN = r"\b(gmbh und co kg|gmbh|e k|ek|kg|ohg|ag|ug)\b"
MONATE = {"januar": 1, "jan": 1, "februar": 2, "feb": 2, "märz": 3, "maerz": 3, "mär": 3,
          "mrz": 3, "april": 4, "apr": 4, "mai": 5, "juni": 6, "jun": 6, "juli": 7, "jul": 7,
          "august": 8, "aug": 8, "september": 9, "sep": 9, "sept": 9, "oktober": 10, "okt": 10,
          "november": 11, "nov": 11, "dezember": 12, "dez": 12}
TEXTSPALTEN = ["quelle", "kunde", "bestelldatum", "betrag", "plz", "ort", "artikelgruppe"]

# Muster für Datumsformate (Gruppen: siehe regel_4)
ISO = r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})"                 # 2025-03-14
DEUTSCH = r"(\d{1,2})\.(\d{1,2})\.(\d{4}|\d{2})"             # 14.03.2025 oder 14.03.25
SCHRAEG = r"(\d{1,2})[/-](\d{1,2})[/-](\d{4}|\d{2})"         # 03/14/2025 oder 14/03/2025


def markiere_zeile(df, zeilenindex, grund):
    """Schreibt einen Hinweis in genau eine Zeile (benutzt markiere aus der Vorlage)."""
    return markiere(df, df.index == zeilenindex, grund)


def schluessel(text):
    """Vergleichsschlüssel für Namen: Kleinschreibung, nur Buchstaben und Ziffern.
    'KFZ-Werkstatt Engel GmbH & Co. KG' -> 'kfz werkstatt engel gmbh und co kg'."""
    t = unicodedata.normalize("NFKC", text).casefold().replace("&", " und ")
    t = re.sub(r"(?<=\b[a-zäöüß])\.(?=[a-zäöüß]\b)", "", t)   # G.m.b.H. -> gmbh, e.K. -> ek
    t = re.sub(r"[^0-9a-zäöüß]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def kern(text):
    """Wie schluessel, aber ohne Rechtsform: 'tischlerei funk e k' -> 'tischlerei funk'."""
    return re.sub(r"\s+", " ", re.sub(RECHTSFORMEN, "", schluessel(text))).strip()


def plz_normal(text):
    """Gibt (PLZ, in_Ordnung) zurück: vierstellig -> führende 0 davor, 'D-34117' -> '34117'."""
    t = re.sub(r"^(DE|D)[\s-]*", "", text.strip(), flags=re.I).replace(" ", "")
    if re.fullmatch(r"\d{5}", t):
        return t, True
    if re.fullmatch(r"\d{4}", t):
        return "0" + t, True
    return text.strip(), text.strip() == ""


def jahr(text):
    return int(text) if len(text) == 4 else 2000 + int(text)


def tag_pruefen(j, mo, tg):
    """Gibt das Datum als JJJJ-MM-TT zurück, wenn es das Datum wirklich gibt."""
    try:
        return datetime.date(j, mo, tg).isoformat(), None
    except ValueError:
        return None, "Datum gibt es nicht (z. B. 31.02.)"


def datum_lesen(text, slash_modus):
    """Gibt (JJJJ-MM-TT, None) zurück oder (None, Grund), wenn das Datum nicht sicher lesbar ist."""
    t = text.strip()
    if t == "":
        return None, "Datum fehlt"
    m = re.fullmatch(ISO, t)
    if m:
        return tag_pruefen(int(m[1]), int(m[2]), int(m[3]))
    m = re.fullmatch(DEUTSCH, t)                      # Punkt bedeutet: Tag.Monat.Jahr
    if m:
        return tag_pruefen(jahr(m[3]), int(m[2]), int(m[1]))
    m = re.fullmatch(SCHRAEG, t)                      # Schrägstrich: Reihenfolge ist offen
    if m:
        a, b, j = int(m[1]), int(m[2]), jahr(m[3])
        if a > 12:                                    # erste Zahl kann kein Monat sein
            return tag_pruefen(j, b, a)
        if b > 12:                                    # zweite Zahl kann kein Monat sein
            return tag_pruefen(j, a, b)
        if slash_modus == "TM":
            return tag_pruefen(j, b, a)
        if slash_modus == "MT":
            return tag_pruefen(j, a, b)
        return None, "Datum nicht eindeutig (Tag/Monat oder Monat/Tag?)"
    m = re.fullmatch(r"(\d{1,2})\.?\s*([A-Za-zÄÖÜäöüß]+)\.?,?\s*(\d{4})", t)   # 5. Oktober 2025
    if m and m[2].casefold() in MONATE:
        return tag_pruefen(int(m[3]), MONATE[m[2].casefold()], int(m[1]))
    m = re.fullmatch(r"([A-Za-zÄÖÜäöüß]+)\.?\s*(\d{4})", t)                    # Februar 2025
    if m and m[1].casefold() in MONATE:
        return None, "Tag fehlt (nur Monat und Jahr bekannt)"
    return None, "Datum nicht lesbar"


def lerne_slash_modus(df):
    """Bestimmt je Quelle, ob Schrägstrich-Daten Tag/Monat ('TM') oder Monat/Tag ('MT') sind.
    Beleg: Eine Zahl größer als 12 kann kein Monat sein. Gibt es Belege für beide
    Reihenfolgen in derselben Quelle (oder keinen), bleibt die Quelle unentschieden."""
    belege = {}
    for quelle, text in zip(df["quelle"], df["bestelldatum"]):
        m = re.fullmatch(SCHRAEG, text.strip())
        if not m:
            continue
        a, b = int(m[1]), int(m[2])
        if a > 12 >= b:
            belege.setdefault(quelle, set()).add("TM")
        if b > 12 >= a:
            belege.setdefault(quelle, set()).add("MT")
    return {q: next(iter(s)) for q, s in belege.items() if len(s) == 1}


def betrag_lesen(text):
    """Gibt ('1250.80', None) zurück oder (None, Grund). Nimmt € und EUR weg, erkennt Dezimalkomma,
    Tausenderpunkt und Tausenderkomma. Das letzte Trennzeichen mit 1 bis 2 Stellen dahinter ist
    das Dezimalzeichen. Genau 3 Stellen dahinter heißt: Tausendertrenner (Euro hat 2 Dezimalstellen)."""
    t = text.strip()
    if t == "":
        return None, "Betrag fehlt"
    uebrig = re.sub(r"€|euro|eur|[\d.,\-\s'’()]", "", t, flags=re.I)    # was bleibt, wenn man Erlaubtes streicht?
    if uebrig or not re.search(r"\d", t):
        return None, "Betrag nicht lesbar"
    negativ = "-" in t or (t.startswith("(") and t.endswith(")"))
    zahl = re.sub(r"[^\d.,]", "", t)
    if re.fullmatch(r"\d+\.\d{6,}", zahl):                       # Rechenrest: 3.7799999999999998
        wert = float(zahl)
        if wert < 10:                    # '3.78' ist als Euro-Betrag unplausibel: gemeint war '3.780' (Tausenderpunkt)
            return ("-" if negativ else "") + f"{round(wert * 1000):.2f}", None
        gerundet = round(wert, 2)
        if abs(gerundet - wert) < 1e-6:
            return ("-" if negativ else "") + f"{gerundet:.2f}", None
    zeichen = re.findall(r"[.,]", zahl)
    if not zeichen:                                              # 406
        ganz, dez = zahl, "00"
    elif re.fullmatch(r"\d+[.,]\d{1,2}", zahl):                  # 117,13 oder 117.13
        ganz, dez = re.split(r"[.,]", zahl)
    elif re.fullmatch(r"\d{1,3}([.,]\d{3})+[.,]\d{1,2}", zahl):  # 1.250,80 oder 1,250.80
        dezzeichen = zeichen[-1]
        if zeichen[:-1].count(dezzeichen) or len(set(zeichen[:-1])) != 1:
            return None, "Betrag nicht eindeutig (Trennzeichen)"
        ganz, dez = zahl[:zahl.rfind(dezzeichen)], zahl[zahl.rfind(dezzeichen) + 1:]
        ganz = re.sub(r"[.,]", "", ganz)
    elif re.fullmatch(r"[1-9]\d{0,2}([.,]\d{3})+", zahl) and len(set(zeichen)) == 1:   # 1.250 oder 12,500
        ganz, dez = re.sub(r"[.,]", "", zahl), "00"
    else:
        return None, "Betrag nicht eindeutig (Trennzeichen)"
    return ("-" if negativ else "") + ganz + "." + dez.ljust(2, "0"), None


def stamm_vorbereiten(stamm):
    """Kundenstamm mit Vergleichsschlüsseln und fünfstelliger PLZ (im Stamm fehlt oft die führende 0)."""
    st = stamm[stamm["kunde"].str.strip() != ""].copy()
    st["k"] = st["kunde"].map(schluessel)
    st["kern"] = st["kunde"].map(kern)
    st["plz5"] = st["plz"].map(lambda p: plz_normal(p)[0])
    return st.reset_index(drop=True)


def finde_kunde(name, plz, ort, st):
    """Sucht den Kunden im Stamm. Gibt (Stammzeile oder None, Hinweis oder None) zurück."""
    k = schluessel(name)
    treffer = st[st["k"] == k]                                   # 1. Stufe: gleich nach Normalisieren
    if treffer.empty and kern(name):
        treffer = st[st["kern"] == kern(name)]                   # 2. Stufe: gleich ohne Rechtsform
    if treffer.empty:
        aehnlich = difflib.get_close_matches(k, st["k"].tolist(), n=1, cutoff=0.85)
        zusatz = ""
        if aehnlich:
            zusatz = f", ähnlich: {st.loc[st['k'] == aehnlich[0], 'kunde'].iloc[0]} (nicht übernommen)"
        return None, "Kunde nicht im Kundenstamm" + zusatz
    if treffer["kundennr"].nunique() > 1:                        # derselbe Name, mehrere Betriebe
        plz_ok = plz_normal(plz)[0]
        eng = treffer[treffer["plz5"] == plz_ok] if plz_ok else treffer.iloc[0:0]
        if len(eng) != 1 and ort:
            eng = treffer[treffer["ort"].map(schluessel) == schluessel(ort)]
        if len(eng) != 1:
            nummern = ", ".join(sorted(treffer["kundennr"]))
            return None, f"Kunde mehrdeutig (Kundenstamm: {nummern}), PLZ/Ort entscheidet nicht"
        treffer = eng
    return treffer.iloc[0], None

## Regel 1 · Komplett leere Zeilen löschen
**Wenn** in einer Zeile **alle** Zellen leer sind, **dann** löschen. Die Leerzeile am Ende der Datei ist nur eine Trennzeile, keine Bestellung. Dasselbe gilt für Export-Müll wie `,,,,;;;;` (ohne echte Bestellnummer). *(Nachtrag nach dem Test auf Datei B.)* (Löschen ist erlaubt, nur die Spalte `zeile` darf sich nie ändern.)

In [ ]:
def regel_1_leerzeilen(df, stamm):
    """Wenn: eine Zeile in allen Spalten (außer hinweis) leer ist.
    Dann: Zeile löschen.   Außer: nichts, denn eine leere Zeile enthält keine Bestellung."""
    inhalt = df[TEXTSPALTEN].apply(lambda s: s.str.strip(" ,;") != "")
    echte_id = df["zeile"].str.fullmatch(r"[A-Z]-\d+")       # ',,,,' in zeile ist Export-Müll, keine Bestellnummer
    return df[inhalt.any(axis=1) | echte_id].copy()

anwenden(regel_1_leerzeilen)

## Regel 2 · Leerzeichen am Rand entfernen
**Wenn** ein Text mit einem Leerzeichen anfängt oder aufhört (oder zwei Leerzeichen hintereinander hat), **dann** werden sie entfernt. Sonst wären `Gartenbau Haase GmbH` und ` Gartenbau Haase GmbH` zwei verschiedene Kunden. Bei Texten enden auch Export-Reste wie `Handwerkzeug,,,` hier. *(Nachtrag: in Datei A gab es das nicht, in B fast in jeder Zeile.)*

In [ ]:
def regel_2_leerzeichen(df, stamm):
    """Wenn: ein Text mit Leerzeichen beginnt oder endet, oder zwei Leerzeichen hintereinander hat.
    Dann: Leerzeichen am Rand entfernen, doppelte zu einem machen.   Außer: Spalte zeile (nie ändern)."""
    for spalte in TEXTSPALTEN:
        df[spalte] = df[spalte].str.strip().str.replace(r"\s{2,}", " ", regex=True)
        if spalte in ("quelle", "kunde", "ort", "artikelgruppe"):          # Export-Rest: 'Handwerkzeug,,,'
            df[spalte] = df[spalte].str.replace(r"[,;]+$", "", regex=True)
    return df

anwenden(regel_2_leerzeichen)

## Regel 3 · Kaputte Umlaute reparieren
**Wenn** ein Text `Ã` enthält (z. B. `GÃ¶ttingen`), **dann** wird er zurückübersetzt (`Göttingen`). Hintergrund: Ein UTF-8-Text wurde irrtümlich als Windows-Text gelesen, aus jedem `ö` wurden zwei Zeichen. Die Umkehrung ist **eindeutig**, deshalb ist sie sicher. Geht sie nicht auf, bleibt der Text stehen und wird markiert.

In [ ]:
def regel_3_umlaute(df, stamm):
    """Wenn: ein Text 'Ã' oder 'â€' enthält (UTF-8-Text, der als Windows-Text gelesen wurde, z. B. 'GÃ¶ttingen').
    Dann: Text zurückübersetzen ('Göttingen').   Außer: geht die Rückübersetzung nicht, Text lassen
    und markieren."""
    for spalte in TEXTSPALTEN:
        for i in df.index[df[spalte].str.contains("Ã|â€|Â", regex=True)]:
            try:
                df.at[i, spalte] = df.at[i, spalte].encode("cp1252").decode("utf-8")
            except (UnicodeEncodeError, UnicodeDecodeError):
                df = markiere_zeile(df, i, f"{spalte}: kaputte Zeichen, nicht reparierbar")
    return df

anwenden(regel_3_umlaute)

## Regel 4 · Datum in `JJJJ-MM-TT` bringen
Die Regel kennt vier Fälle:

| Fall | Beispiel | Ergebnis |
|---|---|---|
| Punkt: Tag.Monat.Jahr | `02.01.2025`, `16.01.25` | `2025-01-02`, `2025-01-16` |
| Schrägstrich, **eine Zahl über 12** | `01/14/2025` | `2025-01-14`: 14 kann kein Monat sein, also ist die zweite Zahl der Tag |
| Schrägstrich, **beide Zahlen ≤ 12** | `01/02/2025` | gilt die Reihenfolge, die **dieselbe Quelle** an anderer Stelle eindeutig gezeigt hat (bei Hoffmann: Monat/Tag). Gibt es keinen Beleg: **markieren** |
| nur Monat, **kein Tag** | `Februar 2025` | bleibt stehen, **markiert** ("Tag fehlt") |

**Warum lernt die Regel die Reihenfolge je Quelle und kennt nicht einfach "Hoffmann"?** Käme im zweiten Halbjahr ein neuer Außendienstler mit Schrägstrich-Daten (Herr Neumann ab Oktober), müsste sonst jemand die Regel anfassen. So findet sie die Reihenfolge selbst, und wenn sie keinen Beleg hat, **rät sie nicht**.

In [ ]:
def regel_4_datum(df, stamm):
    """Wenn: bestelldatum nicht JJJJ-MM-TT ist.
    Dann: in JJJJ-MM-TT umschreiben. Punkt = Tag.Monat.Jahr, zweistelliges Jahr = 20xx. Bei Schrägstrich
    entscheidet eine Zahl über 12 (kann kein Monat sein), sonst die Reihenfolge, die die Quelle
    in diesem Lauf sonst benutzt.   Außer: Reihenfolge unklar, Tag fehlt (nur Monat), Datum nicht lesbar
    oder gibt es nicht: Wert lassen und markieren."""
    modus = lerne_slash_modus(df)
    for i in df.index:
        iso, grund = datum_lesen(df.at[i, "bestelldatum"], modus.get(df.at[i, "quelle"]))
        if iso:
            df.at[i, "bestelldatum"] = iso
        else:
            df = markiere_zeile(df, i, grund)
    return df

anwenden(regel_4_datum)

## Regel 5 · Betrag in `Zahl.Zahl` bringen
Der Betrag soll als Zahl mit **Punkt** und **zwei Stellen** dastehen: `1250.80`. *(Nachtrag nach dem Fehlerprotokoll: `14.100,00 €` kann auch `141,00` gemeint sein, das Komma sitzt falsch. Deshalb wird jeder Außendienst-Betrag ohne Cent (`,00` oder ganz ohne Stellen) ab 5000 **markiert**, der Wert bleibt stehen. Der Webshop ist maschinell erzeugt und wird nicht verdächtigt.)*

*(Nachtrag nach Datei B: Rechenreste wie `3.7799999999999998` sind in Wahrheit `3.780` (Tausenderpunkt) und eingelesen als Zahl 3,78. Ein Betrag unter 10 € ist unplausibel, deshalb wird er mal 1000 genommen: `3780.00`. Ab 10 € wird nur auf zwei Stellen gerundet. Das Fehlerprotokoll von B bestätigt es.)*

| Vorher | Nachher | Warum |
|---|---|---|
| `192,17 €` | `192.17` | Komma ist Dezimalzeichen, `€` weg |
| `406 EUR` | `406.00` | Text `EUR` weg, keine Stellen: `.00` |
| `1.250,80 €` | `1250.80` | Beide Zeichen da: das **letzte** ist das Dezimalzeichen |
| `1,017.62` | `1017.62` | US-Schreibweise: gleiche Logik |
| `-254.65` | `-254.65` | **Minus bleibt**: Gutschrift des Webshops |

**Falle:** Ein einzelnes Zeichen mit **drei** Ziffern dahinter (`1.250`) ist ein Tausendertrenner (Euro hat nur zwei Dezimalstellen). Ist es **nicht eindeutig** (`1.250.50`) oder steht Text darin (`ca. 400`), bleibt der Wert stehen und wird markiert.

In [ ]:
def regel_5_betrag(df, stamm):
    """Wenn: betrag nicht 'Zahl mit Punkt und zwei Stellen' ist (z. B. '192,17 €', '406 EUR', '1.250,80').
    Dann: € und EUR entfernen, Dezimalkomma zu Punkt, Tausenderzeichen entfernen, zwei Dezimalstellen.
    Minus bleibt (Gutschrift!).   Außer: Betrag leer oder Trennzeichen nicht eindeutig: Wert lassen und markieren.
    Zusätzlich markiert (Wert bleibt): Außendienst, glatter Betrag (',00' oder ohne Cent) ab 5000: mögliches Kommafehler."""
    for i in df.index:
        roh = df.at[i, "betrag"]
        neu, grund = betrag_lesen(roh)
        if neu:
            df.at[i, "betrag"] = neu
            # Verdacht Kommafehler: '14.100,00 €' kann 141,00 sein. Nicht sicher: Wert lassen, markieren.
            if df.at[i, "quelle"].startswith("AD") and neu.endswith(".00") and abs(float(neu)) >= 5000:
                df = markiere_zeile(df, i, "Betrag evtl. Kommafehler (Außendienst, glatter Betrag, hoch)")
        else:
            df = markiere_zeile(df, i, grund)
    return df

anwenden(regel_5_betrag)

## Regel 6 · PLZ in Ordnung bringen
Zwei Fälle, beide haben dieselbe Ursache (Datei kommt aus Excel):
- **PLZ steht im Ortsfeld** (`36043 Fulda`): Die PLZ wandert in die PLZ-Spalte, der Ort steht allein.
- **PLZ hat nur vier Ziffern** (`4109`): Excel hat die führende `0` verschluckt (Leipzig ist `04109`). Eine `0` davor setzen, **sicher**, solange es genau vier Ziffern sind.

Passt etwas nicht zusammen (PLZ im Ort **widerspricht** der PLZ-Spalte), bleibt der Wert stehen und wird markiert.

In [ ]:
def regel_6_plz(df, stamm):
    """Wenn: der Ort mit einer PLZ beginnt ('36043 Fulda').
    Dann: PLZ in die Spalte plz (wenn dort leer), Ort ohne PLZ.   Außer: steht in plz eine andere PLZ:
    markieren.
    Wenn: die PLZ vierstellig ist (Excel hat die führende 0 verschluckt: 4109).
    Dann: eine 0 davorsetzen (04109).   Außer: PLZ hat dann nicht 5 Ziffern: Wert lassen und markieren."""
    for i in df.index:
        m = re.fullmatch(r"(\d{4,5})\s+(.+)", df.at[i, "ort"])
        if m:
            plz_im_ort = plz_normal(m[1])[0]
            if df.at[i, "plz"] == "":
                df.at[i, "plz"] = plz_im_ort
            elif plz_normal(df.at[i, "plz"])[0] != plz_im_ort:
                df = markiere_zeile(df, i, f"PLZ im Ort ({plz_im_ort}) widerspricht der PLZ-Spalte ({df.at[i, 'plz']})")
            df.at[i, "ort"] = m[2]
        wert, ok = plz_normal(df.at[i, "plz"])
        df.at[i, "plz"] = wert
        if not ok:
            df = markiere_zeile(df, i, f"PLZ nicht fünfstellig: '{wert}'")
    return df

anwenden(regel_6_plz)

## Regel 7 · Kundenname mit dem Kundenstamm abgleichen
Das Warehouse will den Namen **genau so, wie er im Kundenstamm steht**. Das Skript sucht in zwei Stufen:
*(Nachtrag nach Datei B: `G.m.b.H.` wird vorher zu `GmbH` vereinheitlicht, sonst findet man diese ~35 Zeilen nicht im Stamm.)*

1. **gleich nach Normalisieren** (Groß/Klein, Bindestrich, Punkte egal): `TISCHLEREI ARNOLD GMBH & CO. KG` ist `Tischlerei Arnold GmbH & Co. KG`.
2. **gleich ohne Rechtsform**: `Sanitär Möller` ist `Sanitär Möller e.K.`, **aber nur wenn genau ein Kunde** so heißt.

**Zwei Fallen:**
- **Elektro Weber** gibt es zweimal im Stamm (Kassel und Baunatal). Hier entscheidet die **PLZ** (oder der Ort) der Bestellung. Ist beides leer, bleibt die Zeile **markiert**.
- **Ott** und **Otto** sind **verschiedene** Kunden. Bei Namen, die nur *ähnlich* sind, **ändert das Skript nichts**. Es schreibt den Vorschlag in den Hinweis ("ähnlich: …").

In [ ]:
def regel_7_kunde(df, stamm):
    """Wenn: der Kundenname nicht genau so im Kundenstamm steht.
    Dann: Name aus dem Kundenstamm übernehmen, aber nur wenn genau EIN Kunde passt (gleich nach
    Groß/Klein, Bindestrich, Punkt, oder gleich ohne Rechtsform wie GmbH / e.K.). Bei mehreren
    Betrieben mit gleichem Namen entscheiden PLZ und Ort.
    Außer: kein oder mehrere passende Kunden: Name lassen und markieren. Ähnlich klingende Namen
    (Ott / Otto) werden NIE automatisch zusammengelegt, nur als Vorschlag in den Hinweis geschrieben."""
    st = stamm_vorbereiten(stamm)
    df["_kundennr"] = ""
    for i in df.index:
        name = df.at[i, "kunde"]
        if name == "":
            df = markiere_zeile(df, i, "Kunde fehlt")
            continue
        zeile, grund = finde_kunde(name, df.at[i, "plz"], df.at[i, "ort"], st)
        if zeile is None:
            df = markiere_zeile(df, i, grund)
        else:
            df.at[i, "kunde"] = zeile["kunde"]
            df.at[i, "_kundennr"] = zeile["kundennr"]
    return df

anwenden(regel_7_kunde)

## Regel 8 · Leere PLZ und Ort aus dem Kundenstamm ergänzen
Wenn der Kunde (Regel 7) eindeutig ist und PLZ oder Ort **leer** sind, setzt die Regel die Werte des Kundenstamms ein. Im Kundenstamm fehlt bei 30 von 120 Kunden die führende `0`, auch dort wird sie ergänzt.

**Nicht überschrieben** wird ein **vorhandener anderer** Wert. Er könnte eine andere Lieferadresse sein. Er wird markiert. Auch markiert: Der Kunde ist unbekannt, oder im Stamm ist selbst nichts hinterlegt.

In [ ]:
def regel_8_plz_ort_stamm(df, stamm):
    """Wenn: der Kunde eindeutig im Kundenstamm gefunden wurde (Regel 7) und plz oder ort leer sind.
    Dann: aus dem Kundenstamm einsetzen. Ist der Ort nur anders geschrieben (GÖTTINGEN), Schreibweise
    des Kundenstamms nehmen.   Außer: steht ein anderer Wert drin (PLZ oder Ort weicht vom Stamm ab, zum
    Beispiel eine andere Lieferadresse), Wert lassen und markieren. Auch markieren, wenn der Kunde nicht
    im Stamm ist oder der Stamm selbst leer ist: dann bleibt plz oder ort leer."""
    st = stamm_vorbereiten(stamm).set_index("kundennr")
    for i in df.index:
        nr = df.at[i, "_kundennr"]
        plz, ort = df.at[i, "plz"], df.at[i, "ort"]
        if nr == "":                                              # Kunde unbekannt: nichts einsetzbar
            if plz == "":
                df = markiere_zeile(df, i, "PLZ fehlt (Kunde nicht im Stamm)")
            if ort == "":
                df = markiere_zeile(df, i, "Ort fehlt (Kunde nicht im Stamm)")
            continue
        s = st.loc[nr]
        if plz == "":
            if s["plz5"]:
                df.at[i, "plz"] = s["plz5"]
            else:
                df = markiere_zeile(df, i, "PLZ fehlt (auch im Kundenstamm leer)")
        elif s["plz5"] and plz != s["plz5"]:
            df = markiere_zeile(df, i, f"PLZ weicht vom Kundenstamm ab ({plz} statt {s['plz5']})")
        if ort == "":
            if s["ort"]:
                df.at[i, "ort"] = s["ort"]
            else:
                df = markiere_zeile(df, i, "Ort fehlt (auch im Kundenstamm leer)")
        elif s["ort"] and ort != s["ort"]:
            if schluessel(ort) == schluessel(s["ort"]):
                df.at[i, "ort"] = s["ort"]
            else:
                df = markiere_zeile(df, i, f"Ort weicht vom Kundenstamm ab ({ort} statt {s['ort']})")
    return df

anwenden(regel_8_plz_ort_stamm)

## Regel 9 · Doppelte Bestellungen finden
*(Nachtrag nach dem Fehlerprotokoll von B: Eine Außendienst-Zeile mit Lücken, also ohne Betrag oder nur mit Monat, zählt auch als Dublette, wenn es genau **einen** Webshop-Auftrag desselben Kunden im selben Monat (bzw. am selben Tag) gibt. Sonst würden zwei Zeilen übrig bleiben.)*

Jetzt, wo Kunde, Datum und Betrag **vereinheitlicht** sind, lassen sich Dubletten erkennen. Beispiel: `Glaserei Winter, 17.02.2025, 372,65 €` (Außendienst) und `Glaserei Winter, 2025-02-17, 372.65` (Webshop) sind **dieselbe** Bestellung.

- **Genau eine** der Zeilen kommt aus dem **Webshop**: Die Außendienst-Zeile wird **gelöscht**. Der Webshop ist das Originalsystem, der Außendienst hat nur zusätzlich eingetragen.
- Sonst (zweimal Webshop, oder zwei Außendienstler): **nichts löschen**, nur die spätere Zeile markieren.
- **Keine Dublette:** gleicher Kunde und gleicher Betrag, aber **anderes Datum** (Schreinerei Ott bestellt jeden Monat 312,50 €).

In [ ]:
def regel_9_dubletten(df, stamm):
    """Wenn: zwei Zeilen denselben Kunden, dasselbe Datum und denselben Betrag haben (erst nach den
    Regeln 4 bis 8 erkennbar, vorher sehen sie verschieden aus).
    Dann: ist genau EINE davon aus dem Webshop, die anderen (Außendienst) löschen. Der Webshop ist die
    Quelle, der Außendienst hat die Bestellung nur zusätzlich eingetragen.
    Außer: sonst (zweimal Webshop, zwei Außendienstler) nichts löschen, die späteren Zeilen markieren.
    Nicht betroffen: gleicher Betrag, aber anderes Datum (zum Beispiel jeden Monat dieselben Schleifmittel)."""
    ok = df["bestelldatum"].str.fullmatch(r"\d{4}-\d{2}-\d{2}") & df["betrag"].str.fullmatch(r"-?\d+\.\d{2}") \
        & (df["kunde"] != "")
    schl = df["kunde"].map(schluessel)
    gruppen = df[ok].groupby([schl[ok], df.loc[ok, "bestelldatum"], df.loc[ok, "betrag"]])
    loeschen = []
    for _, g in gruppen:
        if len(g) < 2:
            continue
        webshop = g[g["quelle"].str.casefold() == "webshop"]
        if len(webshop) == 1:
            loeschen += list(g.index.difference(webshop.index))
        else:
            for i in g.index[1:]:
                df = markiere_zeile(df, i, f"mögliche Dublette von {g['zeile'].iloc[0]}")
    # Dubletten mit Lücken: Außendienst-Zeile ohne Betrag oder nur mit Monat, aber genau ein passender Webshop-Auftrag
    rest = df.drop(index=loeschen)
    web = rest[rest["quelle"].str.casefold() == "webshop"]
    webk = web["kunde"].map(schluessel)
    for i in rest.index:
        if not rest.at[i, "quelle"].startswith("AD") or rest.at[i, "kunde"] == "":
            continue
        d, b = rest.at[i, "bestelldatum"], rest.at[i, "betrag"]
        voll = bool(re.fullmatch(r"\d{4}-\d{2}-\d{2}", d))
        m = re.fullmatch(r"(\w+)\.?\s+(\d{4})", d)
        if voll:
            monat = d[:7]
        elif m and m[1].casefold() in MONATE:
            monat = f"{m[2]}-{MONATE[m[1].casefold()]:02d}"
        else:
            continue
        if b != "" and voll:               # nichts fehlt: das hat die Hauptprüfung schon erledigt
            continue
        kand = web[(webk == schluessel(rest.at[i, "kunde"])) & (web["bestelldatum"].str[:7] == monat)]
        if voll:
            kand = kand[kand["bestelldatum"] == d]
        if b != "":
            kand = kand[kand["betrag"] == b]
        if len(kand) == 1:
            loeschen.append(i)
    return df.drop(index=loeschen)

anwenden(regel_9_dubletten)

## Regel 10 · Aufräumen

In [ ]:
def regel_10_aufraeumen(df, stamm):
    """Entfernt die Hilfsspalte _kundennr, die Regel 7 und 8 benutzt haben. Zielformat: keine Extra-Spalten."""
    return df.drop(columns=["_kundennr"])

anwenden(regel_10_aufraeumen)

REGELN_NOTEBOOK = None   # Platzhalter, die Liste steht im Abschnitt „Alle Regeln auf einmal“

## Alle Regeln auf einmal: die Liste `REGELN`
Im Skript `regeln_Mattis.py` stehen die Regeln in dieser Liste. Das Skript führt sie **von oben nach unten** aus, genau wie wir es hier Schritt für Schritt getan haben. Gleich prüfen wir, dass beides **dasselbe Ergebnis** liefert.

In [ ]:
REGELN = [regel_1_leerzeilen, regel_2_leerzeichen, regel_3_umlaute, regel_4_datum, regel_5_betrag,
          regel_6_plz, regel_7_kunde, regel_8_plz_ort_stamm, regel_9_dubletten, regel_10_aufraeumen]

# Frischer Lauf: alle Regeln hintereinander auf die Rohdaten
d_komplett = roh.copy()
d_komplett["hinweis"] = ""
for regel in REGELN:
    d_komplett = regel(d_komplett, stamm)
print("Schritt für Schritt (oben) und alles auf einmal liefern dasselbe:",
      d_komplett.reset_index(drop=True).equals(df.reset_index(drop=True)))

---
## Ergebnis prüfen
### 1 · Wie viele Zeilen sind übrig, und was wurde geändert?

In [ ]:
diff, geloescht = aenderungen_zwischen(roh, df)
print(f"Zeilen: {len(roh)} gelesen, {len(df)} geschrieben, {len(geloescht)} gelöscht")
print("Gelöscht:", geloescht)
print()
print("Geänderte Zellen je Spalte:")
print(diff.groupby("spalte").size().to_string())
print(f"\nZeilen mit Hinweis: {(df.hinweis != '').sum()}")

### 2 · Welche Zeilen sind markiert und warum? (das, was das Skript **nicht sicher** konnte)

In [ ]:
markiert = df[df.hinweis != ""][["zeile", "quelle", "kunde", "bestelldatum", "betrag", "plz", "ort", "hinweis"]]
display(markiert)

gruende = df.hinweis[df.hinweis != ""].str.split(" | ", regex=False).explode()
gruende = gruende.str.replace(r"\s*[(:,].*", "", regex=True).str.strip()      # Grund ohne Zusätze in Klammern
print("Gründe, nach Häufigkeit:")
print(gruende.value_counts().to_string())

### 3 · Stimmt das Ergebnis mit dem Zielformat überein?
Eine **Prüfliste**, jede Zeile heißt: Wie viele Zeilen entsprechen dem Zielformat, und wie viele sind markiert (also bewusst offen)?

In [ ]:
def pruefe(t):
    """Zählt je Zielformat-Regel die Zeilen, die entsprechen / nicht entsprechen."""
    kunden_stamm = set(stamm.kunde)
    stamm5 = stamm.assign(plz5=stamm.plz.map(lambda p: plz_normal(p)[0])).set_index("kunde")
    z = []
    def zeile(name, ok):
        z.append({"Prüfung": name, "ok": int(ok.sum()), "nicht ok": int((~ok).sum()),
                  "davon markiert": int((~ok & (t.hinweis != "")).sum())})
    zeile("bestelldatum ist JJJJ-MM-TT", t.bestelldatum.str.fullmatch(r"\d{4}-\d{2}-\d{2}"))
    zeile("betrag ist Zahl mit Punkt und 2 Stellen", t.betrag.str.fullmatch(r"-?\d+\.\d{2}"))
    zeile("plz ist fünfstellig", t.plz.str.fullmatch(r"\d{5}"))
    zeile("ort ohne PLZ davor", ~t.ort.str.match(r"^\d{4,5} "))
    zeile("keine kaputten Umlaute", ~t.apply(lambda s: s.str.contains("Ã|â€")).any(axis=1))
    zeile("kunde steht im Kundenstamm", t.kunde.isin(kunden_stamm))
    zeile("keine Leerzeichen am Rand", ~t[TEXTSPALTEN].apply(lambda s: s != s.str.strip()).any(axis=1))
    return pd.DataFrame(z)

vorher = roh[roh.zeile.str.strip() != ""].copy()
vorher["hinweis"] = ""
vergleich = pruefe(vorher).merge(pruefe(df), on="Prüfung", suffixes=(" (vorher)", " (nachher)"))
vergleich[["Prüfung", "ok (vorher)", "nicht ok (vorher)", "ok (nachher)", "nicht ok (nachher)", "davon markiert (nachher)"]]

**Wie ist das zu lesen?** Eine Zeile mit **"nicht ok (nachher)" > 0**, die auch **"davon markiert"** hat, ist in Ordnung: Das Skript hat gesagt "hier bin ich nicht sicher". Gefährlich ist **nicht ok ohne Markierung**, das wären stille Fehler. Die Spalte darf dort nicht größer sein als die markierte.

In [ ]:
still = vergleich[vergleich["nicht ok (nachher)"] > vergleich["davon markiert (nachher)"]]
print("Prüfungen mit unmarkierten Resten (stille Fehler):", len(still))
still

### 4 · Zweiter Lauf: ändert sich etwas? (Idempotenz)
Wendet man die Regeln **ein zweites Mal** auf das Ergebnis an, darf sich **nichts** mehr ändern. Das ist ein guter Test: Ein Skript, das jede Nacht läuft, darf bereits saubere Daten nicht verschlimmbessern.

In [ ]:
zweiter = df.copy()
zweiter["hinweis"] = ""            # Hinweise neu sammeln, sonst stünden sie doppelt da
for regel in REGELN:
    zweiter = regel(zweiter, stamm)
daten = [c for c in df.columns if c != "hinweis"]
print("Zweiter Lauf ändert nichts:", zweiter[daten].reset_index(drop=True).equals(df[daten].reset_index(drop=True)))
# Hinweise zählen nicht mit: der Kommafehler-Verdacht erkennt man nur am Rohtext ('14.100,00 €'), den es im Ergebnis nicht mehr gibt.

### 5 · Härtetest mit Daten, die wir nicht kennen
Die Prüfdaten aus dem zweiten Halbjahr sehen wir erst später. Als Ersatz baue ich **kleine Fälle von Hand**, die so in H2 vorkommen könnten (neuer Außendienstler, andere Schreibweisen, Fallen). Zu jedem Fall steht, was **richtig** wäre. Die Zeile `assert` stoppt das Notebook, wenn die Regeln etwas anderes tun.

In [ ]:
n = stamm.kunde.tolist()
kunde_a = stamm.kunde.iloc[3]                                   # irgendein Kunde aus dem Stamm
kunde_b = stamm.kunde.iloc[7]
plz_b = plz_normal(stamm.plz.iloc[7])[0]
ort_b = stamm.ort.iloc[7]

fall = [
 # zeile, quelle,         kunde,                   datum,              betrag,        plz,       ort
 ("T1", "AD Neumann",  kunde_a.upper(),         "10/13/2025",       "EUR 1 250,80", "",        ""),          # Tag 13 -> Monat/Tag
 ("T2", "AD Neumann",  kunde_b,                 "10/05/2025",       "312.5",        plz_b,     ort_b),       # Neumann: aus T1 gelernt: Monat/Tag
 ("T3", "AD Neues",    kunde_b,                 "03/04/2025",       "99,00 €",      plz_b,     ort_b),       # keine Reihenfolge belegt
 ("T4", "Webshop",     kunde_b,                 "5. Oktober 2025",  "1.250",        plz_b,     ort_b),       # Tag + Monatsname, Tausenderpunkt
 ("T5", "Webshop",     kunde_b,                 "31.02.2025",       "ca. 400",      plz_b,     ort_b),       # Datum gibt es nicht, Betrag nicht lesbar
 ("T6", "Webshop",     "Schreinrei Kaiser e.K.", "2025/10/06",      "-20,50",       "D-34117", "Kassel"),    # Tippfehler im Namen, Gutschrift
 ("T7", "Webshop",     kunde_a,                 "2025-10-07",       "100.00",       "",        ""),          # Dublette von T8 (Webshop bleibt)
 ("T8", "AD Becker",   kunde_a,                 "07.10.2025",       "100,00 €",     "",        ""),
 ("T9", "Webshop",     "Völlig Unbekannt GmbH", "2025-10-08",       "55.00",        "37073",   "Göttingen"), # neuer Kunde, nicht im Stamm
]
test = pd.DataFrame(fall, columns=["zeile", "quelle", "kunde", "bestelldatum", "betrag", "plz", "ort"])
test["artikelgruppe"] = "Handwerkzeug"
test = test[["zeile", "quelle", "kunde", "bestelldatum", "betrag", "plz", "ort", "artikelgruppe"]]
test["hinweis"] = ""
for regel in REGELN:
    test = regel(test, stamm)
test = test.set_index("zeile")
display(test[["quelle", "kunde", "bestelldatum", "betrag", "plz", "ort", "hinweis"]])

# Erwartungen: was ist richtig?
assert test.loc["T1", "bestelldatum"] == "2025-10-13" and test.loc["T1", "betrag"] == "1250.80"
assert test.loc["T1", "kunde"] == kunde_a and test.loc["T1", "plz"] != ""           # PLZ/Ort aus dem Stamm ergänzt
assert test.loc["T2", "bestelldatum"] == "2025-10-05" and test.loc["T2", "betrag"] == "312.50"   # aus T1 gelernt
assert test.loc["T3", "bestelldatum"] == "03/04/2025" and "nicht eindeutig" in test.loc["T3", "hinweis"]
assert test.loc["T4", "bestelldatum"] == "2025-10-05" and test.loc["T4", "betrag"] == "1250.00"
assert test.loc["T5", "bestelldatum"] == "31.02.2025" and test.loc["T5", "betrag"] == "ca. 400"   # Finger weg
assert "gibt es nicht" in test.loc["T5", "hinweis"] and "nicht lesbar" in test.loc["T5", "hinweis"]
assert test.loc["T6", "betrag"] == "-20.50" and test.loc["T6", "plz"] == "34117"      # Minus bleibt, D- weg
assert "nicht im Kundenstamm" in test.loc["T6", "hinweis"]                            # Tippfehler nicht geraten
assert "T8" not in test.index and "T7" in test.index                                  # Außendienst-Dublette weg
assert "nicht im Kundenstamm" in test.loc["T9", "hinweis"] and test.loc["T9", "kunde"] == "Völlig Unbekannt GmbH"
print("Alle Erwartungen erfüllt.")

## Was ich weiß, und was nicht
**Gesichert** (an den Daten belegt): Die Zahlen im Abschnitt "Ergebnis prüfen", der zweite Lauf ändert nichts, und die Fälle im Härtetest verhalten sich wie beschrieben.

**Annahmen, die ich getroffen habe** (bitte prüfen):
- **Dubletten:** Ich lösche die **Außendienst-Zeile**, wenn derselbe Eintrag aus dem Webshop da ist. Ob der Webshop wirklich die "richtige" Quelle ist, ist eine Fachfrage, das Aufgabenblatt sagt es nicht. Würde die Messung die andere Zeile erwarten, ist es **eine Zeile Code** (Regel 9).
- **Tausenderpunkt:** `1.250` lese ich als 1250 (Euro hat zwei Dezimalstellen). In der Datei A kommt das nur mit Komma-Teil vor, es ist also eine Annahme für H2.
- **Webshop-Gutschriften** (negative Beträge) bleiben unverändert. Ob sie in H2 anders erfasst sind, weiß ich nicht.

**Was erst die Prüfdaten zeigen:** Ob die Regeln **allgemein** genug sind. Die Hauptgefahr sind **neue Schreibweisen**, die in Datei A nicht vorkamen, zum Beispiel ein neuer Außendienstler ab Oktober. Die Regeln sind dafür so gebaut, dass sie bei Unsicherheit **markieren** statt raten.